# Aria API Reference Guide

**Complete reference for all HTTP endpoints across Azure Functions and Aria web server.**

Use this guide to understand every endpoint, required parameters, response formats, and error cases.


## API Endpoints Overview

### Functions API (Port 7071)

- Chat endpoints: `/api/chat`, `/api/chat/stream`
- Status endpoints: `/api/ai/status`, `/api/quantum-llm/status`
- TTS endpoint: `/api/tts`
- Quantum endpoints: `/api/quantum/*`

### Aria Web Server (Port 8080)

- State endpoints: `/api/aria/state`, `/api/aria/objects`
- Command endpoints: `/api/aria/command`, `/api/aria/execute`
- Schema endpoints: `/api/aria/schema`, `/api/aria/health`
- Object endpoints: `/api/aria/object`
- World endpoint: `/api/aria/world`

---

## Chat API Reference

### POST /api/chat

**Complete chat message (non-streaming)**

```bash
curl -X POST http://localhost:7071/api/chat \
  -H "Content-Type: application/json" \
  -d '{
    "message": "What is quantum computing?",
    "session_id": "user123",
    "temperature": 0.7,
    "max_tokens": 500
  }'
```

**Request Parameters:**

- `message` (string, required): User message
- `session_id` (string, optional): Session ID for memory
- `temperature` (float, optional): 0.0-2.0, default 0.7
- `max_tokens` (integer, optional): Default 500
- `provider` (string, optional): Explicit provider selection

**Response:**

```json
{
    "response": "Quantum computing uses quantum bits...",
    "provider": "azure",
    "tokens_used": 150,
    "session_id": "user123"
}
```

**Error Cases:**

- 400: Invalid request format
- 500: Provider error
- 503: Service unavailable

---

### GET /api/chat/stream (SSE)

**Streaming chat response**

```bash
curl http://localhost:7071/api/chat/stream?message=hello&session_id=user123
```

**Query Parameters:**

- `message` (required): User message
- `session_id` (optional): Session ID
- `temperature` (optional): Default 0.7

**Response Format (Server-Sent Events):**

```
data: {"chunk":"The ","type":"token"}
data: {"chunk":"answer ","type":"token"}
data: {"type":"done"}
```

**Event Types:**

- `token`: Text chunk
- `done`: Stream complete
- `error`: Error occurred

---

## Status API Reference

### GET /api/ai/status

**System health and configuration status**

```bash
curl http://localhost:7071/api/ai/status | jq
```

**Response:**

```json
{
    "provider": "azure",
    "provider_available": true,
    "database": {
        "available": true,
        "pool_size": 10,
        "active_connections": 3
    },
    "embeddings": {
        "available": true,
        "model": "text-embedding-ada-002"
    },
    "azure_openai": {
        "available": true,
        "endpoint": "https://..."
    },
    "openai": {
        "available": false
    },
    "lmstudio": {
        "available": true,
        "url": "http://127.0.0.1:1234"
    },
    "ollama": {
        "available": false
    },
    "cosmos": {
        "available": true
    },
    "uptime_seconds": 3600
}
```

---

### GET /api/quantum-llm/status

**Quantum LLM backend status**

```bash
curl http://localhost:7071/api/quantum-llm/status | jq
```

**Response:**

```json
{
    "provider": "quantum_sampler",
    "backend": "simulator",
    "qubits": 20,
    "fallback_available": true,
    "quantum_available": true,
    "latest_job_id": "quantum_job_123"
}
```

---

## Aria Web API Reference

### GET /api/aria/state

**Current Aria state**

```bash
curl http://localhost:8080/api/aria/state | jq
```

**Response:**

```json
{
    "aria": {
        "position": { "x": 50, "y": 50 },
        "state": "idle",
        "expression": "neutral",
        "looking_at": null
    },
    "objects": [
        {
            "id": "ball_001",
            "name": "ball",
            "position": { "x": 75, "y": 30 },
            "type": "sphere"
        }
    ],
    "environment": {
        "lighting": "bright",
        "time_of_day": "afternoon"
    }
}
```

---

### POST /api/aria/command

**Parse natural language command**

```bash
curl -X POST http://localhost:8080/api/aria/command \
  -H "Content-Type: application/json" \
  -d '{"command": "wave and smile"}'
```

**Request:**

```json
{
    "command": "move left and pick up the ball"
}
```

**Response:**

```json
{
    "command": "move left and pick up the ball",
    "tags": ["aria:position:left", "aria:pickup:ball"],
    "actions": [
        {
            "type": "move",
            "target": "left",
            "duration": 1.0
        },
        {
            "type": "pickup",
            "object": "ball",
            "duration": 0.5
        }
    ],
    "confidence": 0.95
}
```

---

### POST /api/aria/execute

**Execute action sequence**

```bash
curl -X POST http://localhost:8080/api/aria/execute \
  -H "Content-Type: application/json" \
  -d '{
    "actions": [
      {"type": "move", "target": "center"},
      {"type": "say", "text": "Hello!"},
      {"type": "gesture", "gesture": "wave"}
    ],
    "mode": "execute"
  }'
```

**Request Parameters:**

- `actions` (array, required): Action sequence
- `mode` (string, optional): "plan" or "execute", default "execute"

**Action Types:**

- `move`: Move to position
- `say`: Speak text
- `gesture`: Make gesture
- `pickup`: Pick up object
- `drop`: Drop object
- `throw`: Throw object
- `look`: Look at target
- `wait`: Wait duration

**Response:**

```json
{
    "mode": "execute",
    "status": "completed",
    "actions_executed": 3,
    "duration": 2.5,
    "final_state": {
        "position": { "x": 50, "y": 50 },
        "expression": "happy"
    }
}
```

---

### GET /api/aria/schema

**Get valid actions and constraints**

```bash
curl http://localhost:8080/api/aria/schema | jq
```

**Response:**

```json
{
    "actions": {
        "move": {
            "parameters": {
                "target": {
                    "type": "string|coordinates",
                    "options": ["left", "right", "center", "forward", "back"]
                }
            },
            "duration": { "min": 0.1, "max": 10 }
        },
        "gesture": {
            "gestures": ["wave", "bow", "shrug", "clap", "thumbs_up", "nod"]
        },
        "say": {
            "text": { "max_length": 200 }
        }
    },
    "positions": ["left", "right", "center", "forward", "back"],
    "limits": {
        "max_actions": 25,
        "max_say_length": 200,
        "max_wait_duration": 30
    }
}
```

---

### GET /api/aria/health

**Aria server health check**

```bash
curl http://localhost:8080/api/aria/health | jq
```

**Response:**

```json
{
    "status": "healthy",
    "uptime_seconds": 1800,
    "version": "1.0.0",
    "entities": {
        "aria": 1,
        "objects": 5
    },
    "providers": {
        "llm": "available",
        "tts": "available"
    },
    "latest_action": {
        "type": "gesture",
        "timestamp": 1625000000
    }
}
```

---

## TTS API Reference

### POST /api/tts

**Text-to-speech synthesis**

```bash
curl -X POST http://localhost:7071/api/tts \
  -H "Content-Type: application/json" \
  -d '{"text": "Hello, world!"}'
```

**Request:**

```json
{
    "text": "Hello, world!",
    "voice": "en-US-JennyNeural",
    "rate": 1.0
}
```

**Response:** Audio file (WAV or MP3)

**Error Cases:**

- 400: Invalid text
- 413: Text too long (>1000 chars)
- 503: TTS service unavailable

---

## Quantum API Reference

### POST /api/quantum/job

**Submit quantum job**

```bash
curl -X POST http://localhost:7071/api/quantum/job \
  -H "Content-Type: application/json" \
  -d '{
    "circuit": "QASM circuit...",
    "backend": "simulator",
    "shots": 100
  }'
```

**Response:**

```json
{
    "job_id": "quantum_job_123",
    "status": "submitted",
    "backend": "simulator"
}
```

---

### GET /api/quantum/job/{job_id}

**Get quantum job status and results**

```bash
curl http://localhost:7071/api/quantum/job/quantum_job_123 | jq
```

**Response:**

```json
{
    "job_id": "quantum_job_123",
    "status": "completed",
    "results": {
        "00": 45,
        "01": 5,
        "10": 30,
        "11": 20
    }
}
```

---

## Common Response Codes

| Code | Meaning      | Cause                  |
| ---- | ------------ | ---------------------- |
| 200  | OK           | Request succeeded      |
| 400  | Bad Request  | Invalid parameters     |
| 401  | Unauthorized | Missing/invalid auth   |
| 404  | Not Found    | Resource doesn't exist |
| 429  | Rate Limited | Too many requests      |
| 500  | Server Error | Provider/DB error      |
| 503  | Unavailable  | Service down           |

---

## Rate Limiting

**Default Limits:**

- Chat: 10 req/sec per session
- Status: 1 req/sec
- Aria: 20 req/sec

**Headers:**

```
X-RateLimit-Limit: 10
X-RateLimit-Remaining: 8
X-RateLimit-Reset: 1625000060
```

---

## Authentication

**Current:** No authentication required (local development)

**Production:** OAuth2 or API Key required

```bash
# Future: API key in header
curl http://localhost:7071/api/chat \
  -H "Authorization: Bearer your_api_key"
```

---

## Testing API Endpoints

### Test All Endpoints

```bash
#!/bin/bash

# Test chat
curl -X POST http://localhost:7071/api/chat \
  -H "Content-Type: application/json" \
  -d '{"message":"test"}'

# Test status
curl http://localhost:7071/api/ai/status

# Test Aria
curl http://localhost:8080/api/aria/state

# Test schema
curl http://localhost:8080/api/aria/schema
```

### Test with jq (JSON parsing)

```bash
# Pretty print response
curl http://localhost:7071/api/ai/status | jq '.'

# Extract field
curl http://localhost:7071/api/ai/status | jq '.provider'

# Filter results
curl http://localhost:7071/api/ai/status | jq '.database | select(.available == true)'
```

### Test SSE Streaming

```bash
# Test stream with timeout
timeout 5 curl http://localhost:7071/api/chat/stream?message=hello

# Parse events
curl http://localhost:7071/api/chat/stream?message=hello | while read -r line; do
  echo "$line" | jq -R 'fromjson | select(.type=="token") | .chunk' 2>/dev/null || true
done
```


## API Usage Examples

### Example 1: Simple Chat

```python
import requests
import json

response = requests.post(
    'http://localhost:7071/api/chat',
    json={
        'message': 'Hello, how are you?',
        'session_id': 'user_123'
    }
)

data = response.json()
print(f"Response: {data['response']}")
print(f"Provider: {data['provider']}")
```

### Example 2: Streaming Chat with Events

```python
import requests

response = requests.get(
    'http://localhost:7071/api/chat/stream',
    params={
        'message': 'Tell me a story',
        'session_id': 'user_123'
    },
    stream=True
)

for line in response.iter_lines():
    if line:
        event = json.loads(line.split('data: ')[1])
        if event['type'] == 'token':
            print(event['chunk'], end='', flush=True)
        elif event['type'] == 'done':
            print('\n[Done]')
            break
```

### Example 3: Aria Command Execution

```python
import requests

# Parse command
cmd_response = requests.post(
    'http://localhost:8080/api/aria/command',
    json={'command': 'wave and move to the left'}
)

actions = cmd_response.json()['actions']
print(f"Parsed {len(actions)} actions")

# Execute actions
exec_response = requests.post(
    'http://localhost:8080/api/aria/execute',
    json={
        'actions': actions,
        'mode': 'execute'
    }
)

print(f"Execution status: {exec_response.json()['status']}")
```

### Example 4: Check System Health

```python
import requests

response = requests.get('http://localhost:7071/api/ai/status')
status = response.json()

print(f"Provider: {status['provider']}")
print(f"Database: {status['database']['available']}")
print(f"Azure: {status['azure_openai']['available']}")
print(f"LM Studio: {status['lmstudio']['available']}")
```
